# Local RAG, end to end — Ollama + FAISS

Everything in this notebook runs **on your laptop**. No API keys, no cloud, no bill.

By the end you will have built a question-answering system over 13 novels, watched it
give a **confident, cited, completely wrong answer**, worked out why, and fixed it.

## The pipeline we are building

```
                                  ┌─────────────────────────────────┐
                                  │  13 novels  (~1.9M words)       │  Part 2
                                  └────────────────┬────────────────┘
                                                   │  split into 300-word
                                                   ▼  windows, 60-word overlap
                                  ┌─────────────────────────────────┐
                                  │  8,509 chunks                   │  Part 3
                                  └────────────────┬────────────────┘
                                                   │  embeddinggemma
                                                   ▼  768-dim vectors
                                  ┌─────────────────────────────────┐
                                  │  FAISS index (cosine)           │  Part 4
                                  └────────────────┬────────────────┘
                                                   │
   ┌─────────────┐   embed        ┌────────────────▼────────────────┐
   │  question   ├───────────────▶│  top-k nearest chunks           │  Part 5
   └─────────────┘                └────────────────┬────────────────┘
                                                   │
                                  ┌────────────────▼────────────────┐
                                  │  !! this is where it breaks !!  │  Part 6
                                  └────────────────┬────────────────┘
                                                   │  pull in each hit's
                                                   ▼  neighbouring chunks
                                  ┌─────────────────────────────────┐
                                  │  windowed context               │  Part 7
                                  └────────────────┬────────────────┘
                                                   │
                                  ┌────────────────▼────────────────┐
                                  │  LLM answers, with citations    │  Part 8
                                  └────────────────┬────────────────┘
                                                   │
                                  ┌────────────────▼────────────────┐
                                  │  measure it / ship it (Flask)   │  Part 9-10
                                  └─────────────────────────────────┘
```

## Map

| Part | What |
|---|---|
| 0 | Setup — Ollama, dependencies |
| 1 | Embeddings: turning text into vectors |
| 2 | The corpus: 13 public-domain novels |
| 3 | Chunking: 300-word windows |
| 4 | Embedding everything + the FAISS index |
| 5 | Retrieval: top-k nearest neighbours |
| 6 | **Where it breaks** — the best match is not the answer |
| 7 | Windowed retrieval: the fix |
| 8 | Generation: grounded answers with citations |
| 9 | Measuring retrieval instead of trusting it |
| 10 | Shipping it as a web app |

## Before you start

In a terminal:

```bash
ollama list                      # what you already have
ollama pull embeddinggemma       # embedding model  (~600 MB)
ollama pull llama3.1:8b          # generator        (~4.9 GB)
ollama pull qwen3.5:9b           # second generator, for Part 8
```

---
# Part 0 — Setup

```
[ P0 ]  P1   P2   P3   P4   P5   P6   P7   P8   P9   P10
 ^^^^
```

In [1]:
# Everything this notebook needs. Safe to re-run.
%pip -q install ollama faiss-cpu numpy pandas tqdm requests truststore ipython-autotime flask


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# autotime prints the wall-clock time under every cell from here on.
# Useful here: you can see exactly which steps are expensive.
%load_ext autotime

import textwrap


def pretty_print(*args):
    """print(), but wrapped to 80 columns so long model output stays readable."""
    text = " ".join(str(a) for a in args)
    print(textwrap.fill(text, width=80))

time: 189 µs (started: 2026-08-29 18:14:00 +05:30)


## Talking to Ollama

[Ollama](https://ollama.com) runs models locally and exposes an HTTP API on
`http://localhost:11434`. The `ollama` Python package wraps that API.

There are two ways to call it:

```python
import ollama

# (1) module-level helper — reads proxy environment variables
ollama.chat(model="llama3.1:8b", messages=[...])

# (2) explicit Client — pins the host, ignores proxy environment variables
client = ollama.Client(host="http://localhost:11434", trust_env=False)
client.chat(model="llama3.1:8b", messages=[...])
```

**We always use (2), and the reason is `trust_env=False`.**

When a VPN or corporate proxy is active, `HTTP_PROXY` / `HTTPS_PROXY` / `ALL_PROXY`
are set globally. The default client is built on `httpx`, which honours those
variables — so it tries to route your `localhost:11434` request *through the VPN*.
The proxy has no idea what your local Ollama is, and the call fails.

`trust_env=False` tells `httpx` to ignore those variables, so the request goes
straight to localhost. On a machine with no VPN both forms work; on a machine with
one, only the second does. Use the second and never think about it again.

In [3]:
import ollama

client = ollama.Client(host="http://localhost:11434", trust_env=False)

# Which models are actually available? If this errors, Ollama isn't running.
available = [m["model"] for m in client.list()["models"]]
pretty_print("Installed:", ", ".join(sorted(available)))

for needed in ["embeddinggemma:latest", "llama3.1:8b"]:
    mark = "OK  " if needed in available else "MISSING -> ollama pull"
    print(f"  {mark} {needed}")

Installed: embeddinggemma:latest, gemma3:1b, gemma3:27b, gemma3:4b, gemma3n:e4b,
gpt-oss:120b, gpt-oss:20b, llama3.1:8b, nomic-embed-text:latest,
qwen3-embedding:8b, qwen3.5:9b, tinyllama:latest
  OK   embeddinggemma:latest
  OK   llama3.1:8b
time: 337 ms (started: 2026-08-29 18:14:00 +05:30)


## The question this whole notebook is about

Before we build anything, let's ask a small local model a question **with no retrieval
at all** — just whatever it memorised during training.

The question is the title mystery of one of the Sherlock Holmes stories, and it is the
question we will follow all the way through this notebook.

Watch the answer carefully. It may well be right — the story is famous. The point is that
**you have no way to tell**, because there is nothing to check it against.

In [4]:
QUESTION = "What was the speckled band?"

resp = client.chat(
    model="llama3.1:8b",
    messages=[{"role": "user", "content": QUESTION}],
    options={"temperature": 0.0, "num_predict": 200},
)

pretty_print("Q:", QUESTION)
print()
pretty_print("A:", resp["message"]["content"].strip())
print()
pretty_print("(no sources, no passages, no way to check it — hold that thought)")

Q: What was the speckled band?

A: A classic mystery!  The Speckled Band is a short story by Sir Arthur Conan
Doyle, first published in 1892. It is one of the most famous stories in the
Sherlock Holmes canon and is considered one of the greatest detective stories of
all time.  The story revolves around the mysterious death of Helen Stoner, a
young woman who is the sister of a woman named Julia Stoner. Julia had died
under suspicious circumstances a year earlier, and Helen has come to consult
Sherlock Holmes about her own life being threatened.  Helen tells Holmes that
her sister Julia had died after a terrifying experience, in which she had been
awakened in the middle of the night by a strange, whistling sound, followed by a
sudden and inexplicable death. Helen believes that her own life is in danger, as
she has heard the same whistling sound and has reason to believe that her own
death is being planned.  Holmes agrees to investigate and, with the help of Dr.
John Watson, sets out

(no

---
# Part 1 — Embeddings: text as vectors

```
 P0  [ P1 ]  P2   P3   P4   P5   P6   P7   P8   P9   P10
      ^^^^
```

An **embedding model** maps a piece of text to a fixed-length list of numbers such that
texts that *mean* similar things land close together.

"Close together" is measured with **cosine similarity** — the cosine of the angle between
two vectors, from -1 (opposite) through 0 (unrelated) to 1 (identical direction).

If we first scale every vector to length 1 (**L2-normalise**), then cosine similarity is
just the dot product. That is a single, very fast instruction — which is exactly why we
normalise before putting anything into FAISS.

In [5]:
import numpy as np

EMBED_MODEL = "embeddinggemma:latest"

resp = client.embeddings(model=EMBED_MODEL, prompt="The quick brown fox jumps over the lazy dog.")
vec = np.asarray(resp["embedding"], dtype="float32")

pretty_print("Model:    ", EMBED_MODEL)
pretty_print("Dimension:", vec.shape)             # 768 for embeddinggemma
pretty_print("First 8:  ", np.round(vec[:8], 4).tolist())
pretty_print("L2 norm:  ", round(float(np.linalg.norm(vec)), 6))

Model:     embeddinggemma:latest
Dimension: (768,)
First 8:   [-0.11029999703168869, 0.05389999970793724, 0.06880000233650208,
-0.022299999371170998, -0.08060000091791153, 0.00419999985024333,
0.03519999980926514, 0.051600001752376556]
L2 norm:   1.0
time: 111 ms (started: 2026-08-29 18:14:04 +05:30)


In [6]:
# Does "similar meaning" really mean "close vector"? Let's check.

def embed_text(text: str) -> np.ndarray:
    """Embed one string and scale it to unit length, so dot product == cosine."""
    v = np.asarray(
        client.embeddings(model=EMBED_MODEL, prompt=text)["embedding"],
        dtype="float32",
    )
    return v / (np.linalg.norm(v) + 1e-12)


a = embed_text("A dog chases a cat in the garden.")
b = embed_text("In the yard, a puppy is running after a kitten.")
c = embed_text("The Fourier transform decomposes a signal into frequencies.")

print(f"cos(a, b)  same idea, no shared words = {float(a @ b):.3f}")
print(f"cos(a, c)  unrelated                   = {float(a @ c):.3f}")

cos(a, b)  same idea, no shared words = 0.787
cos(a, c)  unrelated                   = 0.217
time: 161 ms (started: 2026-08-29 18:14:04 +05:30)


## One detail that is easy to miss: task prefixes

EmbeddingGemma was trained with **instruction prefixes**. The model card asks you to wrap
text differently depending on whether it is a stored document or an incoming search query:

| role | format |
|---|---|
| document | `title: {title} \| text: {content}` |
| query | `task: search result \| query: {content}` |

This is called an **asymmetric** embedding model: a question and the passage that answers
it do not look alike, so the model is told which side it is embedding.

Run the next cell and notice something uncomfortable: on this pair the prefixed score is
**lower** than the unprefixed one.

That is not a bug, and it is worth sitting with for a moment:

> **The absolute cosine value is not a quality score.** Only the *ordering* it induces over
> your corpus matters. Two different formats produce two different number lines, and you
> cannot compare a score on one to a score on the other.

We use the prefixes because the model card specifies them. We judge whether they helped in
**Part 9**, by measuring retrieval, not by admiring a similarity number. And in **Part 7** we
throw out a tempting piece of logic that makes exactly this mistake — comparing a score to a
fixed threshold as if 0.35 meant something universal.

In [7]:
question = "Where does Sherlock Holmes live?"
passage  = ("I had called upon my friend Sherlock Holmes at his rooms in Baker Street, "
            "where he was deep in a chemical investigation.")

plain    = float(embed_text(question) @ embed_text(passage))
prefixed = float(
    embed_text(f"task: search result | query: {question}")
    @ embed_text(f"title: Adventures of Sherlock Holmes | text: {passage}")
)

print(f"no prefixes    {plain:.3f}")
print(f"with prefixes  {prefixed:.3f}")

no prefixes    0.529
with prefixes  0.484
time: 215 ms (started: 2026-08-29 18:14:05 +05:30)


---
# Part 2 — The corpus

```
 P0   P1  [ P2 ]  P3   P4   P5   P6   P7   P8   P9   P10
           ^^^^
```

Thirteen public-domain novels from Project Gutenberg — about 1.9 million words. Big enough
that retrieval is a real problem, small enough to index during a coffee break.

Each download is ~2-3 seconds. If you are behind a VPN and the downloads hang, copy the
pre-stripped copies from `corpus_jupyter_bkp/` into `corpus_jupyter/` and re-run.

In [ ]:
# ── Configuration — everything tunable lives here ────────────────────────
EMBED_MODEL     = "embeddinggemma:latest"
GEN_MODEL       = "llama3.1:8b"      # Part 8 swaps this out; one line to change

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
TOPK            = 5

CORPUS_DIR    = "corpus_jupyter"
ARTIFACTS_DIR = "rag_artifacts"

GUTENBERG_BOOKS = {
    "Moby-Dick":                     "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice":           "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein":                  "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula":                       "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities":          "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby":              "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace":                 "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre":                     "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray":    "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment":          "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights":             "https://www.gutenberg.org/files/768/768-0.txt",
}

time: 415 µs (started: 2026-08-29 18:14:05 +05:30)


In [9]:
import re
from pathlib import Path

import pandas as pd
import requests
import truststore
from tqdm import tqdm

truststore.inject_into_ssl()   # use the OS certificate store (corporate MITM proxies)

# Gutenberg wraps each book in a licence header and footer. Strip them.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)

docs = []
for title, url in GUTENBERG_BOOKS.items():
    out_path = Path(CORPUS_DIR) / f"{title.replace(' ', '_')}.txt"

    if not out_path.exists():
        try:
            raw = requests.get(url, timeout=60)
            raw.raise_for_status()
            text = raw.text
            start, end = START_MARK.search(text), END_MARK.search(text)
            if start and end and end.start() > start.end():
                text = text[start.end():end.start()]
            out_path.write_text(text.strip(), encoding="utf-8")
            print(f"  downloaded  {title}")
        except Exception as e:
            print(f"  FAILED      {title}: {e}")
            continue

    docs.append({
        "title": title,
        "text":  out_path.read_text(encoding="utf-8", errors="ignore"),
        "path":  str(out_path),
    })

print(f"\n{len(docs)} books loaded")


13 books loaded
time: 331 ms (started: 2026-08-29 18:14:05 +05:30)


In [10]:
pd.DataFrame([
    {"title": d["title"], "words": len(d["text"].split())}
    for d in docs
]).sort_values("words", ascending=False).reset_index(drop=True)

,title,words
0,War and Peace,563286
1,Moby-Dick,212796
2,Crime and Punishment,203505
3,Jane Eyre,185390
4,Dracula,161321
5,A Tale of Two Cities,135886
6,Pride and Prejudice,127360
7,Wuthering Heights,115945
8,Adventures of Sherlock Holmes,107562
9,The Picture of Dorian Gray,78979


time: 67.8 ms (started: 2026-08-29 18:14:05 +05:30)


---
# Part 3 — Chunking

```
 P0   P1   P2  [ P3 ]  P4   P5   P6   P7   P8   P9   P10
                ^^^^
```

We cannot embed a whole novel as one vector — a single 768-dim vector cannot represent
200,000 words, and we could not fit the result in a prompt anyway. So we cut each book into
**chunks** and embed those.

Two knobs:

- **`WORDS_PER_CHUNK = 300`** — big enough to hold a complete idea, small enough that a
  handful fit in a prompt.
- **`OVERLAP_WORDS = 60`** — consecutive chunks share their last/first 60 words, so a
  sentence that straddles a boundary still appears whole somewhere.

```
words:  |------------------ 300 ------------------|
chunk 0 |------------------------------------------|
chunk 1                          |------------------------------------------|
                                 |<-- 60 -->|
                                   overlap        step = 300 - 60 = 240
```

Overlap is insurance against a bad cut. It is **not** a substitute for the neighbour
expansion we build in Part 7 — Part 6 shows exactly why.

In [11]:
chunks = []          # each: id, title, text, preview, source_path, chunk_index
step = WORDS_PER_CHUNK - OVERLAP_WORDS

for d in docs:
    words = d["text"].split()
    chunk_index = 0
    for i in range(0, len(words), step):
        segment = words[i:i + WORDS_PER_CHUNK]
        if len(segment) < 75:        # drop the runt at the end of a book
            break
        text = " ".join(segment)
        chunks.append({
            "id":          f"{d['title'].replace(' ', '_')}#chunk{chunk_index}",
            "title":       d["title"],
            "text":        text,
            "preview":     text[:400],
            "source_path": d["path"],
            "chunk_index": chunk_index,
        })
        chunk_index += 1

print(f"{len(chunks)} chunks from {len(docs)} books")
print(f"average {np.mean([len(c['text'].split()) for c in chunks]):.0f} words per chunk")

8509 chunks from 13 books


average 300 words per chunk
time: 193 ms (started: 2026-08-29 18:14:05 +05:30)


In [12]:
# What does the overlap actually look like? Last 12 words of one chunk,
# first 12 of the next — they should be the same text.
a, b = chunks[10], chunks[11]
pretty_print("chunk 10 ends:  ...", " ".join(a["text"].split()[-12:]))
pretty_print("chunk 11 starts:   ", " ".join(b["text"].split()[:12]), "...")

chunk 10 ends:  ... are whale and sturgeon. And these, when either thrown ashore
or caught
chunk 11 starts:    the Nantucket Whale-Fishery_. “Spain—a great whale stranded
on the shores of Europe.” ...
time: 340 µs (started: 2026-08-29 18:14:05 +05:30)


---
# Part 4 — Embedding everything, and the FAISS index

```
 P0   P1   P2   P3  [ P4 ]  P5   P6   P7   P8   P9   P10
                     ^^^^
```

Now the expensive step: 8,509 chunks through the embedding model.

**This takes about 5-6 minutes** on an M-series Mac with 6 parallel workers (measured:
~26 chunks/second on real prose). Two things make it survivable:

- **Threads.** Each call is network-bound on `localhost`, so threads help even though
  Python has a GIL. 6 workers took it from ~68 ms to ~18 ms per chunk. Beyond 6 the
  Ollama server itself becomes the bottleneck — 12 workers measured no faster.
- **Retries.** One dropped connection at chunk 6,000 should not cost you the whole run.

Note we submit `title: … | text: …` — the document-side prefix from Part 1.

In [13]:
import time
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 6
RETRIES     = 3


def embed_text(text: str) -> np.ndarray:
    """Embed one string, unit-normalised, with retries. Used everywhere below."""
    for attempt in range(RETRIES):
        try:
            v = np.asarray(
                client.embeddings(model=EMBED_MODEL, prompt=text)["embedding"],
                dtype="float32",
            )
            return v / (np.linalg.norm(v) + 1e-12)
        except Exception:
            if attempt == RETRIES - 1:
                raise
            time.sleep(0.6 * (attempt + 1))

time: 359 µs (started: 2026-08-29 18:14:05 +05:30)


In [14]:
# ~5-6 minutes for 8,509 chunks. Good moment for a break.
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    emb_vectors = list(tqdm(
        pool.map(lambda c: embed_text(f'title: {c["title"]} | text: {c["text"]}'), chunks),
        total=len(chunks),
        desc=f"embedding ({EMBED_MODEL})",
    ))

emb = np.vstack(emb_vectors)
print("embeddings:", emb.shape)


embedding (embeddinggemma:latest):   0%|          | 0/8509 [00:00<?, ?it/s]


embedding (embeddinggemma:latest):   0%|          | 2/8509 [00:00<10:57, 12.94it/s]


embedding (embeddinggemma:latest):   0%|          | 11/8509 [00:00<02:53, 48.87it/s]


embedding (embeddinggemma:latest):   0%|          | 17/8509 [00:00<02:53, 48.86it/s]


embedding (embeddinggemma:latest):   0%|          | 23/8509 [00:00<02:49, 49.94it/s]

...


embedding (embeddinggemma:latest): 100%|█████████▉| 8501/8509 [05:24<00:00, 24.39it/s]


embedding (embeddinggemma:latest): 100%|█████████▉| 8504/8509 [05:24<00:00, 24.67it/s]


embedding (embeddinggemma:latest): 100%|█████████▉| 8507/8509 [05:24<00:00, 24.90it/s]


embedding (embeddinggemma:latest): 100%|██████████| 8509/8509 [05:24<00:00, 26.22it/s]

embeddings: (8509, 768)
time: 5min 24s (started: 2026-08-29 18:14:05 +05:30)


## The index

`IndexFlatIP` is FAISS's **exact** inner-product index — it compares the query against every
stored vector. No approximation, no training, nothing to tune. At 8,509 vectors that is a
sub-millisecond matrix multiply.

The "IP" is why we normalised: **inner product on unit vectors *is* cosine similarity.**

At millions of vectors you would switch to an approximate index (`IndexIVFFlat`, `IndexHNSW`)
and trade a little recall for a lot of speed. At this size that would be a pure loss —
you would add tuning knobs and lose exactness for no gain.

In [15]:
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("FAISS index:", index.ntotal, "vectors of dim", emb.shape[1])

FAISS index: 8509 vectors of dim 768
time: 28.6 ms (started: 2026-08-29 18:19:30 +05:30)


---
# Part 5 — Retrieval

```
 P0   P1   P2   P3   P4  [ P5 ]  P6   P7   P8   P9   P10
                          ^^^^
```

Embed the question with the **query-side** prefix, then ask FAISS for the nearest chunks.

In [16]:
query = "Best detective in the world"

q_vec = embed_text(f"task: search result | query: {query}")
D, I = index.search(q_vec.reshape(1, -1), TOPK)

print(f"Top {TOPK} for: {query!r}\n")
for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
    c = chunks[idx]
    print(f"  #{rank}  {score:.3f}  {c['title']} (chunk {c['chunk_index']})")
    pretty_print("      ", c["preview"][:300])
    print()

Top 5 for: 'Best detective in the world'

  #1  0.413  Adventures of Sherlock Holmes (chunk 63)
       confidence in Mr. Holmes, sir,” said the police agent loftily. “He has
his own little methods, which are, if he won’t mind my saying so, just a little
too theoretical and fantastic, but he has the makings of a detective in him. It
is not too much to say that once or twice, as in that business of the

  #2  0.410  Adventures of Sherlock Holmes (chunk 132)
       corresponds with the injuries. There is no sign of any other weapon.”
“And the murderer?” “Is a tall man, left-handed, limps with the right leg, wears
thick-soled shooting-boots and a grey cloak, smokes Indian cigars, uses a cigar-
holder, and carries a blunt pen-knife in his pocket. There are severa

  #3  0.400  Adventures of Sherlock Holmes (chunk 134)
       a strong presumption that the person whom McCarthy expected to meet him
at Boscombe Pool was someone who had been in Australia.” “What of the rat,
then?” Sherlock Holme

Sensible: every hit is from *The Adventures of Sherlock Holmes*, and we never told the system
who Sherlock Holmes is or that the books have genres. That is semantic search working.

Now look at the **scores**: they are all around 0.37, and they are all within a few
thousandths of each other. Remember that. It is the subject of Part 7.

---
# Part 6 — Where it breaks

```
 P0   P1   P2   P3   P4   P5  [ P6 ]  P7   P8   P9   P10
                               ^^^^
```

That last query was easy: the answer was a *topic*, and topics are what embeddings are good at.

Here is the question we asked in Part 0, now with a corpus behind it:

> *What was the speckled band?*

Run it and look at the `score` column and the `names_the_answer` column before reading on.

In [ ]:
q_vec = embed_text(f"task: search result | query: {QUESTION}")
D, I = index.search(q_vec.reshape(1, -1), TOPK)

display(pd.DataFrame([{
    "rank":             rank,
    "score":            round(float(score), 3),
    "title":            chunks[idx]["title"],
    "chunk":            chunks[idx]["chunk_index"],
    "names_the_answer": "swamp adder" in chunks[idx]["text"].lower(),
} for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1)]))

# The table says WHICH rank. Read rank 1 in full to see why it is the wrong one.
print("=" * 78)
print("RANK 1 — the best match FAISS could find")
print("=" * 78)
pretty_print(chunks[int(I[0][0])]["text"])

,rank,score,title,chunk,names_the_answer
0,1,0.337,Adventures of Sherlock Holmes,281,False
1,2,0.304,Adventures of Sherlock Holmes,256,False
2,3,0.298,Adventures of Sherlock Holmes,282,True
3,4,0.292,Adventures of Sherlock Holmes,245,False
4,5,0.286,War and Peace,259,False


RANK 1 — the best match FAISS could find
I have ever listened. It swelled up louder and louder, a hoarse yell of pain and
fear and anger all mingled in the one dreadful shriek. They say that away down
in the village, and even in the distant parsonage, that cry raised the sleepers
from their beds. It struck cold to our hearts, and I stood gazing at Holmes, and
he at me, until the last echoes of it had died away into the silence from which
it rose. “What can it mean?” I gasped. “It means that it is all over,” Holmes
answered. “And perhaps, after all, it is for the best. Take your pistol, and we
will enter Dr. Roylott’s room.” With a grave face he lit the lamp and led the
way down the corridor. Twice he struck at the chamber door without any reply
from within. Then he turned the handle and entered, I at his heels, with the
cocked pistol in my hand. It was a singular sight which met our eyes. On the
table stood a dark-lantern with the shutter half open, throwing a brilliant beam
of light u

## Read that table again

Two things are wrong here.

**First, the top hit does not contain the answer.** Rank 1 is the climax of the story — the
scream in the night, then Roylott dead in his chair. Read its last few lines, printed under the
table:

> *"Round his brow he had a peculiar yellow band, with brownish speckles, which seemed to be
> bound tightly round his head."*

That is a description of the thing, in the right story, in the right scene, and it reads like an
answer. It is also wrong: what looks like a band of cloth is a snake. The chunk that says so —
the one naming the **swamp adder** — is at rank 3.

Why? Because we asked "what **was** the speckled band?" and embeddings match on topic, not on
the difference between a definition and a description. The scene and the explanation are about
the same thing, so they look alike in vector space.

> **The best-matching chunk and the chunk containing the answer are not the same thing.**

**Second, look at the scores.** The top hit is **0.337** — the lowest top-1 score anywhere in
this notebook (compare 0.66 for the mad tea party). The whole top-5 spans about 0.05.

That is the retriever quietly telling you it is struggling — but only if you already knew what
a good score looks like *for this model, on this corpus, with these prefixes*. On its own the
number means nothing, which is the entire problem with thresholding on it.

In [18]:
# Where is the answer, then? Look at the top hit's immediate neighbours in the book.
top = int(I[0][0])
hit = chunks[top]

print(f"top hit: {hit['title']}, chunk {hit['chunk_index']}\n")
for offset in (-1, 0, +1):
    neighbour = next(
        (c for c in chunks
         if c["source_path"] == hit["source_path"]
         and c["chunk_index"] == hit["chunk_index"] + offset),
        None,
    )
    if neighbour is None:
        continue
    label = "THE HIT" if offset == 0 else f"offset {offset:+d}"
    print(f"  {label:9}  chunk {neighbour['chunk_index']}  "
          f"names_the_answer={'swamp adder' in neighbour['text'].lower()!s}")

# The chunk immediately after the hit is the one that answers the question. Read it.
after = next(c for c in chunks
             if c["source_path"] == hit["source_path"]
             and c["chunk_index"] == hit["chunk_index"] + 1)

print("\n" + "=" * 78)
print(f"CHUNK {after['chunk_index']} — the very next 300 words")
print("=" * 78)
pretty_print(after["text"].replace("swamp adder", ">>> SWAMP ADDER <<<"))

top hit: Adventures of Sherlock Holmes, chunk 281

  offset -1  chunk 280  names_the_answer=False
  THE HIT    chunk 281  names_the_answer=False
  offset +1  chunk 282  names_the_answer=True

CHUNK 282 — the very next 300 words
noticed during the day. His chin was cocked upward and his eyes were fixed in a
dreadful, rigid stare at the corner of the ceiling. Round his brow he had a
peculiar yellow band, with brownish speckles, which seemed to be bound tightly
round his head. As we entered he made neither sound nor motion. “The band! the
speckled band!” whispered Holmes. I took a step forward. In an instant his
strange headgear began to move, and there reared itself from among his hair the
squat diamond-shaped head and puffed neck of a loathsome serpent. “It is a >>>
SWAMP ADDER <<<!” cried Holmes; “the deadliest snake in India. He has died
within ten seconds of being bitten. Violence does, in truth, recoil upon the
violent, and the schemer falls into the pit which he digs for another. L

There it is.

Both chunks describe the yellow band — chunk 282 opens with the same sentences chunk 281 ends
with, which is the 60-word overlap from Part 3 doing its job. Only chunk 282 keeps going, to
the serpent rearing out of the dead man's hair and Holmes naming it.

The answer was 300 words away the entire time. We just never asked for it.

Notice what the overlap did and did not do. It carried the *description* across the boundary —
that is exactly what overlap is for, protecting a sentence that straddles a split. It did
nothing for the *explanation*, which lives entirely in the next chunk. No overlap size short of
duplicating whole chunks would have fixed this.

That is what the rest of this notebook is about.

---
# Part 7 — Windowed retrieval

```
 P0   P1   P2   P3   P4   P5   P6  [ P7 ]  P8   P9   P10
                                    ^^^^
```

The fix: whenever we retrieve a chunk, also take the chunks **on either side of it in the
original book**.

```
        retrieved by FAISS
               │
    ┌──────────▼──────────┐
    │  161  │  162  │  163  │      <- what we actually send to the LLM
    └───────┴───────┴───────┘
      ±1 neighbours, same book, in reading order
```

To do that in O(1) we need one lookup table from `(book, chunk number)` to a position in
`chunks`. Without it, every neighbour lookup would scan all 8,509 chunks — which is what
the slow loop in the previous cell was doing.

In [19]:
KEY_TO_IDX = {(c["source_path"], c["chunk_index"]): i for i, c in enumerate(chunks)}
print(f"{len(KEY_TO_IDX)} (book, chunk) keys")

8509 (book, chunk) keys
time: 1.49 ms (started: 2026-08-29 18:19:30 +05:30)


In [20]:
def expand_with_neighbors_simple(I, chunks, neighbors=1, max_out=8):
    """
    For each FAISS hit in I[0], emit the hit plus its ±neighbors chunks from the
    same book, in reading order. Deduplicated (two hits can share a neighbour),
    hit order preserved, capped at max_out chunks.

    Returns a flat list — one dict per chunk, so each becomes one numbered,
    citable passage in the prompt.
    """
    seen, contexts = set(), []

    for gi in I[0]:
        c = chunks[int(gi)]
        for delta in range(-neighbors, neighbors + 1):
            j = KEY_TO_IDX.get((c["source_path"], c["chunk_index"] + delta))
            if j is None or j in seen:
                continue
            seen.add(j)
            n = chunks[j]
            contexts.append({
                "title":        n["title"],
                "source_path":  n["source_path"],
                "chunk_index":  n["chunk_index"],
                "text":         n["text"],
                "approx_words": len(n["text"].split()),
                "is_hit":       delta == 0,
            })
            if len(contexts) >= max_out:
                return contexts
    return contexts

time: 498 µs (started: 2026-08-29 18:19:30 +05:30)


## A heuristic we are deliberately *not* keeping

A tempting shortcut is to pick how many hits to expand with a confidence rule:

```python
init_topk = 1 if (top1 >= 0.35 and margin >= 0.05) else min(3, topk)
```

The idea is reasonable — *if the best hit is strong and clearly beats the runner-up, trust it
alone; otherwise cast a wider net.* The next cell measures whether it ever actually fires.

In [21]:
probe_questions = [
    "Why did Mr Darcy's first marriage proposal to Elizabeth fail?",
    "Who ran away with Mr Wickham?",
    "What is the name of Ahab's ship?",
    "What does the Queen of Hearts shout when angry?",
    "Who does Alice meet at the mad tea party?",
    "What was the speckled band?",
    "Where does Sherlock Holmes live?",
    "What creature does Victor Frankenstein create?",
]

rows = []
for q in probe_questions:
    D_, _ = index.search(embed_text(f"task: search result | query: {q}").reshape(1, -1), TOPK)
    top1, top2 = float(D_[0][0]), float(D_[0][1])
    rows.append({
        "question": q[:44],
        "top1":     round(top1, 3),
        "margin":   round(top1 - top2, 4),
        "fires?":   top1 >= 0.35 and (top1 - top2) >= 0.05,
    })

probe = pd.DataFrame(rows)
display(probe)
print(f"\nrule fires on {probe['fires?'].sum()} of {len(probe)} questions; "
      f"mean margin {probe['margin'].mean():.4f} against a 0.05 threshold")

,question,top1,margin,fires?
0,Why did Mr Darcy's first marriage proposal t,0.543,0.0007,False
1,Who ran away with Mr Wickham?,0.462,0.0147,False
2,What is the name of Ahab's ship?,0.562,0.0060,False
3,What does the Queen of Hearts shout when ang,0.453,0.0743,True
4,Who does Alice meet at the mad tea party?,0.662,0.1008,True
5,What was the speckled band?,0.337,0.0328,False
6,Where does Sherlock Holmes live?,0.568,0.0071,False
7,What creature does Victor Frankenstein creat,0.570,0.0127,False



rule fires on 2 of 8 questions; mean margin 0.0311 against a 0.05 threshold
time: 541 ms (started: 2026-08-29 18:19:30 +05:30)


The rule fires on **2 of 8** questions (mean margin ~0.031 against a 0.05 threshold), and the
two that fire are the two *easy* ones — the mad tea party and the Queen of Hearts, where any
sensible pipeline would have succeeded regardless. On everything hard it silently falls through
to `min(3, topk)`. It reads like adaptive logic and behaves like a hard-coded 3.

Worse, look at which question has the smallest margin of all: the speckled band, at 0.0007.
The rule cannot distinguish "two passages are equally good" from "two passages are equally
useless".

The deeper lesson is about the `0.35` too:

> **Absolute cosine thresholds do not transfer.** They depend on the embedding model, on
> whether you used its task prefixes (Part 1 showed the number moving *down* when we followed
> the model card), and on the corpus. A number tuned on one setup is meaningless on another.

So we delete the heuristic and pass an explicit `topk`. If you later want adaptive retrieval,
build it on something that survives a model swap — the *shape* of the score curve, or a
reranker that actually reads the passages.

In [22]:
def search_windowed(query: str, topk: int = 3, neighbors: int = 1, max_out: int = 12):
    """
    Embed the query -> FAISS top-k -> expand each hit with ±neighbours.
    Returns (df_hits, contexts): the raw hits for you, the contexts for the LLM.
    """
    q_vec = embed_text(f"task: search result | query: {query}")
    D, I = index.search(q_vec.reshape(1, -1), max(topk, TOPK))

    contexts = expand_with_neighbors_simple(
        np.array([I[0][:topk]]), chunks, neighbors=neighbors, max_out=max_out
    )

    df_hits = pd.DataFrame([{
        "rank":        rank,
        "score":       round(float(score), 3),
        "title":       chunks[idx]["title"],
        "chunk_index": chunks[idx]["chunk_index"],
    } for rank, (score, idx) in enumerate(zip(D[0].tolist(), I[0].tolist()), start=1)])

    return df_hits, contexts

time: 387 µs (started: 2026-08-29 19:56:51 +05:30)


In [23]:
df_hits, contexts = search_windowed(QUESTION, topk=3, neighbors=1)

print("=== raw FAISS hits ===")
display(df_hits)

print(f"\n=== expanded context: {len(contexts)} chunks ===")
display(pd.DataFrame([{
    "title":       c["title"],
    "chunk_index": c["chunk_index"],
    "is_hit":      c["is_hit"],
    "words":       c["approx_words"],
    "names_the_answer": "swamp adder" in c["text"].lower(),
} for c in contexts]))

=== raw FAISS hits ===


,rank,score,title,chunk_index
0,1,0.337,Adventures of Sherlock Holmes,281
1,2,0.304,Adventures of Sherlock Holmes,256
2,3,0.298,Adventures of Sherlock Holmes,282
3,4,0.292,Adventures of Sherlock Holmes,245
4,5,0.286,War and Peace,259



=== expanded context: 7 chunks ===


,title,chunk_index,is_hit,words,names_the_answer
0,Adventures of Sherlock Holmes,280,False,300,False
1,Adventures of Sherlock Holmes,281,True,300,False
2,Adventures of Sherlock Holmes,282,False,300,True
3,Adventures of Sherlock Holmes,255,False,300,False
4,Adventures of Sherlock Holmes,256,True,300,False
5,Adventures of Sherlock Holmes,257,False,300,False
6,Adventures of Sherlock Holmes,283,False,300,False


time: 65.2 ms (started: 2026-08-29 19:56:51 +05:30)


Three hits became seven chunks — fewer than nine, because neighbouring windows overlap and
the duplicates are dropped. Chunk 282 is in the context, so **the swamp adder is finally in
front of the model**.

One column is worth a second look. `is_hit` reads `False` for chunk 282 — but chunk 282 is
**rank 3** in the table right above it. The window got there first: chunk 282 was emitted as
chunk 281's `+1` neighbour, and the dedup kept that first copy. So `is_hit` records *how a
chunk entered the context*, not whether it was also a hit in its own right.

At `topk=1` the distinction collapses and the window does all of the work — rank 3 is never
sent at all, and chunk 282 arrives purely because it sits next to rank 1. That is the
configuration Part 8 opens with.

Whether the model *uses* it is a separate question — which is Part 8.

---
# Part 8 — Generation: the "G" in RAG

```
 P0   P1   P2   P3   P4   P5   P6   P7  [ P8 ]  P9   P10
                                         ^^^^
```

Everything so far was **retrieval**. Now we hand the passages to a model and ask for an
answer that cites them.

Three rules in the prompt, and each one is doing a job:

1. **"Use ONLY the passages"** — the difference between answering *from the corpus* and
   answering from training data. Without it you cannot tell which one you got.
2. **"Cite the passage number"** — makes the answer checkable.
3. **"Say so if the passages don't answer it"** — gives the model an exit that is not
   making something up.

We number the passages and label each with its book and chunk number, so the model can see
which ones are adjacent.

In [24]:
def answer(question: str, contexts: list, model: str = GEN_MODEL, **kw) -> str:
    """Ask `model` to answer `question` using only `contexts`, with citations."""
    block = "\n\n".join(
        f"[{i}] ({c['title']}, chunk {c['chunk_index']})\n{c['text']}"
        for i, c in enumerate(contexts, start=1)
    )
    resp = client.chat(
        model=model,
        messages=[{"role": "user", "content":
            "Answer the question using ONLY the passages below. Cite the passage numbers "
            "you used, like [2].\n"
            "The passages are excerpts from novels, so the speaker of a line may be named "
            "in a neighbouring passage rather than the one containing the line — read them "
            "together before deciding who is speaking.\n"
            "If the passages genuinely do not answer the question, say so and state what "
            "they do show instead.\n\n"
            f"{block}\n\nQuestion: {question}\nAnswer:"
        }],
        options={"temperature": 0.0, "num_predict": 250},
        **kw,
    )
    return resp["message"]["content"].strip()

time: 438 µs (started: 2026-08-29 18:19:31 +05:30)


### Panel A — the top hit alone, no window

This is the naive pipeline: retrieve the single best chunk, hand it over.

In [25]:
_, ctx_a = search_windowed(QUESTION, topk=1, neighbors=0)
print(f"context: {len(ctx_a)} chunk, {sum(c['approx_words'] for c in ctx_a)} words\n")
pretty_print(answer(QUESTION, ctx_a))

context: 1 chunk, 300 words



The passage does not explicitly describe what the speckled band is, but it does
mention that it is a "peculiar yellow band, with brownish speckles" that is
bound tightly around Dr. Roylott's head [1].
time: 3.75 s (started: 2026-08-29 18:19:31 +05:30)


**Look at what just happened.**

The model answered that the speckled band was a *yellow band with brownish speckles bound
around Dr Roylott's head* — **and it cited a passage**. The citation is real. The passage
exists. The quote is accurate.

And the answer is wrong. The speckled band is a **snake**. What the model described is the
snake coiled in a dead man's hair, which the passage describes without ever explaining.
It has confidently reported a murder weapon as an item of clothing.

> **A citation proves a passage was used. It does not prove the passage contained the answer.**

This is what makes bad RAG dangerous rather than merely useless: it does not look like a
failure. There is a real quote, a real source, and a fluent answer. Nothing in the output
flags it.

### Panel B — the same hit, with its neighbours

In [26]:
_, ctx_b = search_windowed(QUESTION, topk=1, neighbors=1)
print(f"context: {len(ctx_b)} chunks, {sum(c['approx_words'] for c in ctx_b)} words\n")
pretty_print(answer(QUESTION, ctx_b))

context: 3 chunks, 900 words



The speckled band was a yellow band with brownish speckles, bound tightly around
Dr. Roylott's head, which was actually a loathsome swamp adder, the deadliest
snake in India, that had been hidden in his hair. [3]
time: 2.71 s (started: 2026-08-29 18:19:35 +05:30)


**Same retriever. Same model. Same question. Same top hit.**

The only change was ±1 chunk of context — and the answer went from an item of clothing to
*"a loathsome swamp adder, the deadliest snake in India"*.

This is the whole argument for windowed retrieval in one cell. The retriever was never really
broken: it found the right *neighbourhood* on its very first try. It just handed over the
wrong 300 words of it.

### Panel C — more hits, still windowed

Panel B worked. So more context should work even better, right? Let's take top-3 hits with
±1 neighbours — seven chunks instead of three.

In [27]:
_, ctx_c = search_windowed(QUESTION, topk=3, neighbors=1)
print(f"context: {len(ctx_c)} chunks, {sum(c['approx_words'] for c in ctx_c)} words\n")
pretty_print(answer(QUESTION, ctx_c))

context: 7 chunks, 2100 words



The passages do not explicitly state what the "speckled band" is. However, based
on the context and the reactions of the characters, it can be inferred that the
"speckled band" refers to a snake, specifically a swamp adder, which is
mentioned in passage [3]. The characters' horror and loathing at the sight of
the snake, as well as the fact that it is described as the "deadliest snake in
India", suggest that the "speckled band" is a snake.  Passage [2] mentions the
"band" and the "speckled band" in the same context, and Holmes' reaction to
seeing the snake is to exclaim "The band! the speckled band!" This suggests that
the "band" refers to the snake's markings or pattern, which is described as a
"peculiar yellow band, with brownish speckles".  Therefore, while the passages
do not explicitly state what the "speckled band" is, the context and the
reactions of the characters suggest that it is a snake, specifically a swamp
adder.
time: 6.4 s (started: 2026-08-29 18:19:37 +05:30)


**More context made the answer worse.**

Panel C hedges — "the passages do not explicitly state… it can be inferred" — even though it
is looking at strictly more information than Panel B, including the exact chunk that names the
adder. The four extra chunks are all *about* the speckled band without defining it, and they
dilute the one chunk that matters.

> **Context is not free. Padding the prompt with plausible-but-unhelpful passages measurably
> degrades the answer.** This is why Part 9 measures instead of assuming.

### Panel D — same context, different model

The right passage **is** in `ctx_c` — we put it there, and Panel B proves the model can use it.
So Panel C is no longer a retrieval problem, and no amount of retriever tuning will fix it.

Let's hold the context fixed and change only the generator.

In [28]:
# think=False is mandatory for reasoning models on Ollama. Without it, qwen3.5 and
# gpt-oss spend the entire num_predict budget on hidden reasoning tokens and return
# an EMPTY string — measured 16.3s of nothing at num_predict=220.
pretty_print("qwen3.5:9b →", answer(QUESTION, ctx_c, model="qwen3.5:9b", think=False))

qwen3.5:9b → The speckled band was a swamp adder, which is described as "the
deadliest snake in India." [3]
time: 11.4 s (started: 2026-08-29 18:19:44 +05:30)


Same passages, same prompt — and a crisp, correct, cited answer where `llama3.1:8b` hedged.

That is the lesson of Part 8, and it is easy to get backwards:

| symptom | actual stage at fault |
|---|---|
| right passage never appears in context | **retriever** — chunking, top-k, window |
| right passage is in context, answer still wrong or hedged | **generator** — model, prompt |

Before tuning a retriever, always check whether the answer was already in the context.
Half the time you are debugging the wrong half of the system.

**On model choice.** `llama3.1:8b` is the default because it is a reasonable, widely-available
8B model and it fails in instructive ways. `qwen3.5:9b` with `think=False` is measurably better
at extracting a specific fact from a noisy context. `GEN_MODEL` at the top is one line — change
it and re-run the panels.

### Back to where we started

Part 0 asked this same question with no retrieval at all. Scroll up and compare that answer
with Panel D's.

The un-retrieved answer may well have been *correct* — the story is famous enough to be
memorised in the weights. That is exactly the trap. You had no way to tell the difference
between the model knowing and the model guessing, because there was nothing attached to it.

Panel D's answer points at a chunk you can open and read.

**RAG's product is not correctness. It is checkability.**

---
# Part 9 — Measure it, don't trust it

```
 P0   P1   P2   P3   P4   P5   P6   P7   P8  [ P9 ]  P10
                                              ^^^^
```

We have been eyeballing single queries. That is how you build a demo, not a system.

The cheapest useful evaluation: a handful of questions where you know the answer, plus a
string that **must** appear in the retrieved context for the answer to be derivable at all.

This measures **context recall** — did retrieval put the necessary evidence in front of the
model? It deliberately does not grade the model's prose. It isolates the retriever, which is
what we are tuning.

Twelve questions is not a benchmark. It is enough to stop you shipping a regression, and you
can write it in ten minutes.

In [29]:
GOLD = [
    ("Why did Mr Darcy's first marriage proposal to Elizabeth fail?", "Pride and Prejudice",           "gentlemanlike"),
    ("Who ran away with Mr Wickham?",                                 "Pride and Prejudice",           "Lydia"),
    ("What does Captain Ahab nail to the mast as a reward?",          "Moby-Dick",                     "doubloon"),
    ("What is the name of Ahab's ship?",                              "Moby-Dick",                     "Pequod"),
    ("What does the Queen of Hearts shout when angry?",               "Alice in Wonderland",           "off with"),
    ("Who does Alice meet at the mad tea party?",                     "Alice in Wonderland",           "Hatter"),
    ("What was the speckled band?",                                   "Adventures of Sherlock Holmes",  "swamp adder"),
    ("Where does Sherlock Holmes live?",                              "Adventures of Sherlock Holmes",  "Baker Street"),
    ("What creature does Victor Frankenstein create?",                "Frankenstein",                  "creature"),
    ("Who is the vampire hunter that pursues Dracula?",               "Dracula",                       "Van Helsing"),
    ("What does Sydney Carton do at the end?",                        "A Tale of Two Cities",          "far, far better"),
    ("What keeps Dorian Gray from ageing?",                           "The Picture of Dorian Gray",    "portrait"),
]

# Embed each gold question once and reuse across the whole sweep.
gold_hits = {
    q: index.search(embed_text(f"task: search result | query: {q}").reshape(1, -1), 5)
    for q, _, _ in GOLD
}
print(f"{len(GOLD)} gold questions embedded")

12 gold questions embedded
time: 720 ms (started: 2026-08-29 18:19:55 +05:30)


In [30]:
rows = []
for topk in (1, 3, 5):
    for neighbors in (0, 1, 2):
        found, sizes = 0, []
        for question, book, must_contain in GOLD:
            D_, I_ = gold_hits[question]
            ctx = expand_with_neighbors_simple(
                np.array([I_[0][:topk]]), chunks, neighbors=neighbors, max_out=100
            )
            sizes.append(len(ctx))
            found += any(must_contain.lower() in c["text"].lower() for c in ctx)
        rows.append({
            "top_k":        topk,
            "neighbors":    neighbors,
            "recall":       f"{found}/{len(GOLD)}",
            "avg_chunks":   round(np.mean(sizes), 1),
            "avg_words":    int(np.mean(sizes) * WORDS_PER_CHUNK),
        })

pd.DataFrame(rows).pivot(index="top_k", columns="neighbors", values=["recall", "avg_chunks"])

recall             avg_chunks            
neighbors      0     1     2          0     1     2
top_k                                              
1           6/12  8/12  8/12        1.0   3.0   5.0
3           9/12  9/12  9/12        3.0   8.3  13.7
5           9/12  9/12  9/12        5.0  13.3  21.3

time: 17 ms (started: 2026-08-29 18:19:56 +05:30)


## Reading the table

Measured on this corpus (your run should reproduce this):

| top-k | ±0 | ±1 | ±2 |
|---|---|---|---|
| **1** | 6/12  (1.0 chunks) | **8/12**  (3.0) | 8/12  (5.0) |
| **3** | 9/12  (3.0) | 9/12  (8.3) | 9/12  (13.7) |
| **5** | 9/12  (5.0) | 9/12  (13.3) | 9/12  (21.3) |

Three results worth more than any single demo:

**1. Neighbours matter most when you retrieve least.** At top-1 the window takes you from
6/12 to 8/12 — the speckled band is one of those two, so Part 6 was not a lucky anecdote.
By top-3 the window adds nothing at all.

**2. More hits beat wider windows.** Going 1→3 on `top_k` buys 3 questions. Going ±0→±1 on a
single hit buys 2, and costs the same 3 chunks. If you have a fixed context budget,
**spend it on more hits before you spend it on wider windows.**

**3. ±2 is never worth it.** It never beat ±1 on any row, and it costs 60% more context.

That last one is why you measure. "More context is better" sounds obviously true, and on this
corpus it is false past ±1 — you would have paid for it in latency and tokens forever without
ever noticing. Panel C is the same finding in prose: the extra chunks made the answer *worse*.

## And the three it never finds

Three questions fail at *every* setting: **Ahab's doubloon**, **Baker Street**, and
**Sydney Carton's last line**. Those are not window problems — the right chunk never enters
the top 5 at all, so no amount of expansion can rescue them.

All three are the same shape: a question whose answer hinges on a rare proper noun or an exact
phrase. Dense embeddings compress exactly that kind of detail away.

The standard fixes are **BM25 keyword search fused with dense retrieval**, and a
**cross-encoder reranker** over a wider candidate set. Both are the natural next step —
and now you have a gold set to prove whether they helped.

---
# Part 10 — Ship it

```
 P0   P1   P2   P3   P4   P5   P6   P7   P8   P9  [ P10 ]
                                                   ^^^^^
```

Nobody is going to run a notebook to ask a question. We save the expensive artifacts to disk
and serve them from a small Flask app, so startup is a file read instead of a 6-minute rebuild.

In [31]:
import json

Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

with open(Path(ARTIFACTS_DIR) / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False)

np.save(Path(ARTIFACTS_DIR) / "embeddings.npy", emb)
faiss.write_index(index, str(Path(ARTIFACTS_DIR) / "faiss_index.bin"))

with open(Path(ARTIFACTS_DIR) / "config.json", "w") as f:
    json.dump({
        "EMBED_MODEL":     EMBED_MODEL,
        "GEN_MODEL":       GEN_MODEL,
        "WORDS_PER_CHUNK": WORDS_PER_CHUNK,
        "OVERLAP_WORDS":   OVERLAP_WORDS,
        "TOPK":            TOPK,
    }, f, indent=2)

for f in sorted(Path(ARTIFACTS_DIR).iterdir()):
    print(f"  {f.name:20} {f.stat().st_size/1e6:8.2f} MB")

  chunks.json             19.30 MB
  config.json              0.00 MB
  embeddings.npy          26.14 MB
  faiss_index.bin         26.14 MB
time: 131 ms (started: 2026-08-29 18:19:56 +05:30)


## The web app

The app lives beside this notebook as two ordinary files, not inside it:

| file | what it is |
|---|---|
| `v3_hello_rag.py` | Flask server — loads `rag_artifacts/`, serves `/` and `/ask` |
| `templates/v3_index.html` | the page — verdict line, rank table, context cards, prompt viewer |

Open them in your editor alongside this notebook. There is nothing in either file you have not
already built here: `embed_text`, `expand_with_neighbors_simple` and `search_windowed` are
copied across **unchanged**, and `/ask` assembles exactly the prompt from Part 8.

Keeping them identical is deliberate. If the app carried its own slightly different copy of
`expand_with_neighbors_simple`, the app and this notebook could quietly disagree about what
"context" means, and every number here would stop describing what the app actually does.

The page adds one control the notebook does not have: a **highlight** box. Type a phrase —
`swamp adder` — and every retrieved chunk is searched for it, case-insensitively. Matches are
highlighted and counted, so you can see which rank holds the answer, which ranks were cut by
top-k, and whether the phrase reached the model at all.

It is a measurement, not a constraint. The phrase never enters the prompt and never touches
retrieval; it is the same after-the-fact substring check as `must_contain` in Part 9.

## Running it

The cell above wrote `rag_artifacts/`, which is everything the app needs. Start it from a
terminal — not from this notebook, since it blocks until you stop it:

```bash
python v3_hello_rag.py
```

Then open <http://localhost:5001>.

The page shows **the answer, the passages it cites, and the exact prompt that produced it**,
all on one screen. The "show the exact prompt" toggle is the point: it is where RAG stops
being magic and becomes string concatenation into a context window.

### What to demo, in order

1. Ask *"What was the speckled band?"* with highlight `swamp adder`, **top-k = 1,
   neighbours = 0**. The answer comes back confident and wrong — a band around a man's head —
   and the verdict reads *the model never saw it*: rank 1 does not contain the phrase, and
   top-k = 1 cut the rank that does.
2. Change **neighbours to 1** and ask again. It becomes a snake, and the verdict flips to *the
   model saw it — but no hit it sent contained it*. A neighbouring chunk carried it. Nothing
   else changed.
3. Set **top-k = 3**. The same passage is now tagged **rank 3** instead of "neighbour" — it
   arrives on its own merit — and the greyed "cut by top-k" rows show what the narrower
   setting had been throwing away.
4. Open **"Show the exact prompt"**. This is where RAG stops being magic: it is a string with
   some retrieved text pasted into it.
5. Untick **"generate an answer"** to get retrieval-only latency, and compare.

---

# What we built

```
question ─▶ embed ─▶ FAISS top-k ─▶ ±neighbour window ─▶ prompt ─▶ answer + citations
```

Every piece is replaceable, and now you can measure whether a replacement helped.

### The five things worth remembering

1. **The best-matching chunk is not the chunk with the answer.** Embeddings rank by topic, and
   the most dramatic passage about a thing routinely outranks the one that explains it.
2. **A citation is not a correctness check.** Panel A quoted a real passage, accurately, and
   still called a snake an item of clothing.
3. **Absolute similarity thresholds do not transfer** across models, prefixes, or corpora.
   0.337 was a bad score here and would be a fine score elsewhere.
4. **More context is not free.** Panel C got worse than Panel B by adding four chunks.
5. **Diagnose the right stage.** If the answer was already in the context, the retriever is
   not your problem.
6. **Twelve gold questions** told us ±2 neighbours is never worth it. Eyeballing never would.

### Where to go next

- **Hybrid retrieval** — BM25 + dense, fused with RRF. Fixes the proper-noun misses in Part 9.
- **Reranking** — retrieve 50, let a cross-encoder pick 5.
- **Better chunking** — split on sentence and paragraph boundaries instead of a word count.